In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler, label_binarize

import warnings
warnings.filterwarnings('ignore')

In [ ]:
RANDOM_STATE = 42
TO_DROP = [
    'StartTime',
    'LastTime',
    # NOTE: src and dst cannot remove for graph probably
    'SrcAddr',
    'DstAddr',
    'sIpId',
    'dIpId',
    # drop target because we are predicting traffic
    'Target'
]

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/annaamalaiu/wustl-iiot-2021-dataset/wustl_iiot_2021.csv")
df.head()

In [ ]:
df["StartTime"] = pd.to_datetime(df["StartTime"])
df["LastTime"] = pd.to_datetime(df["LastTime"])

negative_duration_mask = df['StartTime'] > df['LastTime']
df['StartTime'], df['LastTime'] = (
    np.where(negative_duration_mask, df['LastTime'], df['StartTime']),
    np.where(negative_duration_mask, df['StartTime'], df['LastTime'])
)

df = df.sort_values("StartTime").reset_index(drop=True)
df["nearest_sec"] = df["StartTime"].dt.floor("s")

In [ ]:
df.head()

In [ ]:
proto_dummies = pd.get_dummies(
    df["Proto"],
    prefix="proto",
    dtype=int
)

df = pd.concat([df, proto_dummies], axis=1)

In [ ]:
stos_dummies = pd.get_dummies(
    df['sTos'],
    prefix='stos',
    dtype=int
)

df = pd.concat([df, stos_dummies], axis=1)

In [ ]:
df.head()

In [ ]:
df["Dport"].value_counts()

In [ ]:
def port_range(port):
    if port <= 1023:
        return "well_known"
    elif port <= 49151:
        return "registered"
    else:
        return "dynamic"

# NOTE: the dataset does not contain all the ports checked by the function below
# so the aggregation only takes those columns that are created by the one hot encoding
def dst_port_service(port):
    if port in [20, 21]:
        return "ftp"
    elif port == 22:
        return "ssh"
    elif port == 23:
        return "telnet"
    elif port == 53:
        return "dns"
    elif port in [80, 8080]:
        return "http"
    elif port == 443:
        return "https"
    elif port == 445:
        return "smb"
    elif port == 3389:
        return "rdp"
    elif port == 502:
        return "modbus"
    elif port <= 1023:
        return "other_well_known"
    elif port <= 49151:
        return "registered"
    else:
        return "dynamic"

src_port_features = pd.get_dummies(
    df['Sport'].apply(port_range),
    prefix='src_port',
    dtype=int
)

dst_port_features = pd.get_dummies(
    df["Dport"].apply(dst_port_service),
    prefix="dport",
    dtype=int
)

df = pd.concat([df, src_port_features], axis=1)
df = pd.concat([df, dst_port_features], axis=1)

In [ ]:
per_second = (
    df.groupby(["SrcAddr", "DstAddr", "nearest_sec"])
      .agg(
        mean=("Mean", "sum"),

        # instead of aggregating all the unique source and dest ports, take the number of unique ports used instead
        num_src_ports=('Sport', "nunique"),
        num_dst_ports=('Dport', "nunique"),
        src_port_dynamic=("src_port_dynamic", "max"),
        src_port_registered=("src_port_registered", "max"),
        src_port_well_known=("src_port_well_known", "max"),
        dport_dynamic=("dport_dynamic", "max"),
        dport_ftp=("dport_ftp", "max"),
        dport_http=("dport_http", "max"),
        dport_https=("dport_https", "max"),
        dport_modbus=("dport_modbus", "max"),
        dport_other_well_known=("dport_other_well_known", "max"),
        dport_registered=("dport_registered", "max"),

        src_pkts=("SrcPkts", "sum"),
        dst_pkts=("DstPkts", "sum"),
        tot_pkts=("TotPkts", "sum"),

        src_bytes=("SrcBytes", "sum"),
        dst_bytes=("DstBytes", "sum"),
        tot_bytes=("TotBytes", "sum"),

        src_load=("SrcLoad", "sum"),
        dst_load=("DstLoad", "sum"),
        tot_load=("Load", "sum"),

        src_rate=("SrcRate", "sum"),
        dst_rate=("DstRate", "sum"),
        tot_rate=("Rate", "sum"),

        src_loss=("SrcLoss", "sum"),
        dst_loss=("DstLoss", "sum"),
        loss=("Loss", "sum"),

        # recompute pLoss later

        proto_0=('proto_0', 'max'),
        proto_1=('proto_1', 'max'),
        proto_2=('proto_2', 'max'),
        proto_6=('proto_6', 'max'),
        proto_17=('proto_17', 'max'),
        proto_58=('proto_58', 'max'),
        proto_2054=('proto_2054', 'max'),
        proto_35020=('proto_35020', 'max'),
        
        duration_mean=("Dur", "mean"),
        duration_min=("Dur", "min"),
        duration_max=("Dur", "max"),
        duration_sum=("Dur", "sum"),

        tcp_rtt_mean=('TcpRtt', 'mean'),

        idle_mean=('IdleTime', 'mean'),

        src_dsb_mean=("sDSb", "mean"),
        src_ttl_mean=("sTtl", "mean"),
        dst_ttl_mean=("dTtl", "mean"),

        src_app_bytes=("SAppBytes", "sum"),
        dst_app_bytes=("DAppBytes", "sum"),
        tot_app_bytes=("TotAppByte", "sum"),

        syn_ack_mean=("SynAck", "mean"),

        run_time_sum=("RunTime", "sum"),

        stos_0=("stos_0", "max"),
        stos_16=("stos_16", "max"),
        stos_126=("stos_126", "max"),
        stos_192=("stos_192", "max"),
        stos_207=("stos_207", "max"),

        src_jitter_mean=("SrcJitter", "mean"),
        dst_jitter_mean=("DstJitter", "mean"),

        src_int_pkt_mean=("SIntPkt", "mean"),
        dst_int_pkt_mean=("DIntPkt", "mean"),
        

        # src_rate_mean=("SrcRate", "mean"),
        # dst_rate_mean=("DstRate", "mean"),
        # rate_mean=("Rate", "mean"),

        # src_load_mean=("SrcLoad", "mean"),
        # dst_load_mean=("DstLoad", "mean"),

        # src_jitter_mean=("SrcJitter", "mean"),
        # dst_jitter_mean=("DstJitter", "mean"),

        # tcp_rtt_mean=("TcpRtt", "mean"),
        traffic=("Traffic", "first")
      )
      .reset_index()
)

In [ ]:
per_second

In [ ]:
numeric_features = [
    'mean',
    'num_src_ports',
    'num_dst_ports',
    'src_pkts',
    'dst_pkts',
    'tot_pkts',
    'src_bytes',
    'dst_bytes',
    'tot_bytes',
    'src_load',
    'dst_load',
    'tot_load',
    'src_rate',
    'dst_rate',
    'tot_rate',
    'src_loss',
    'dst_loss',
    'loss',
    'duration_mean',
    'duration_min',
    'duration_max',
    'duration_sum',
    'tcp_rtt_mean',
    'idle_mean',
    'src_dsb_mean',
    'src_ttl_mean',
    'dst_ttl_mean',
    'src_app_bytes',
    'dst_app_bytes',
    'tot_app_bytes',
    'syn_ack_mean',
    'run_time_sum',
    'src_jitter_mean',
    'dst_jitter_mean',
    'src_int_pkt_mean',
    'dst_int_pkt_mean'
]

categorical_features = [
    'proto_0', 'proto_1', 'proto_2', 'proto_6',
    'proto_17', 'proto_58', 'proto_2054', 'proto_35020',
    'stos_0', 'stos_16', 'stos_126', 'stos_192', 'stos_207',
    "src_port_dynamic", "src_port_registered", "src_port_well_known",
    "dport_dynamic", "dport_ftp", "dport_http", "dport_https","dport_modbus",
    "dport_other_well_known", "dport_registered"
]

feature_cols = numeric_features.copy() + categorical_features.copy()

target_col = 'traffic'

In [ ]:
def resample_pair(group):

    group = group.sort_values("nearest_sec")
    group = group.set_index("nearest_sec")

    full_index = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="1s"
    )

    group = group.reindex(full_index)
    group.index.name = "nearest_sec"
    return group

In [ ]:
PRIORITY = ['Backdoor', 'CommInj', 'Reconn', 'DoS', 'normal']

def extract_y(y_seq: np.ndarray):
    # case 1: all nan
    if pd.isna(y_seq).all():
        return "normal"

    labels = y_seq[~pd.isna(y_seq)]
    unique_labels = np.unique(labels)

    # return highest PRIORITY label present, lower index means higher PRIORITY
    for label in PRIORITY:
        if label in unique_labels:
            return label

    return "normal"

In [ ]:
SEQUENCE_LENGTH = 5

X = []
y = []
y_full = []
pairs = []

for (src, dst), group in per_second.groupby(["SrcAddr", "DstAddr"]):
    group = group.sort_values("nearest_sec")
    resampled_group = resample_pair(group)

    features = resampled_group[feature_cols]
    labels = resampled_group["traffic"].values

    # fill null values for features here
    # since the categorical features are one-hot encoded, all 0 for that feature denotes not present
    features[numeric_features] = features[numeric_features].fillna(0)
    features[categorical_features] = features[categorical_features].fillna(0)
    features = features.values

    # pad with zeros if the group is smaller than sequence length
    if len(resampled_group) < SEQUENCE_LENGTH:
        pad_length = SEQUENCE_LENGTH - len(resampled_group)

        features = np.pad(
            features,
            pad_width=((0, pad_length), (0, 0)),
            mode="constant",
            constant_values=0
        )

    for i in range(0, len(resampled_group) - SEQUENCE_LENGTH + 1, SEQUENCE_LENGTH):
        X_seq = features[i:i + SEQUENCE_LENGTH]
        y_seq = labels[i: i + SEQUENCE_LENGTH]
        y_val = extract_y(y_seq)

        X.append(X_seq)
        y.append(y_val)
        y_full.append(y_seq)
        pairs.append((src, dst))

X = np.array(X)
y = np.array(y)
y_full = np.array(y_full)
pairs = np.array(pairs)

# 70/10/20 split
X_train, X_test, y_train, y_test, y_full_train, y_full_test, pairs_train, pairs_test = train_test_split(
    X, y, y_full, pairs, test_size=0.2,
    random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val, y_full_train, y_full_val, pairs_train, pairs_val = train_test_split(
    X_train, y_train, y_full_train, pairs_train, test_size=0.125,
    random_state=RANDOM_STATE, stratify=y_train
)

In [ ]:
# get all indices corresponding to windows with at least 1 attack and 1 normal second; only augmenting train set
attack_idxs = [idx for idx, val in enumerate(y_train) if val != "normal" and pd.isna(y_full_train[idx]).any()]
print(len(attack_idxs))
print(len(y_train))

In [ ]:
# use all windows with at least 1 attack and 1 normal second for augmentation
idxs_to_augment = attack_idxs

In [ ]:
n_numeric = len(numeric_features)

# categorical_features = [
#     'proto_0', 'proto_1', 'proto_2', 'proto_6',
#     'proto_17', 'proto_58', 'proto_2054', 'proto_35020',
#     'stos_0', 'stos_16', 'stos_126', 'stos_192', 'stos_207',
#     "src_port_dynamic", "src_port_registered", "src_port_well_known",
#     "dport_dynamic", "dport_ftp", "dport_http", "dport_https","dport_modbus",
#     "dport_other_well_known", "dport_registered"
# ]

proto_end = 8
proto_num = 8

stos_end = 13
stos_num = 5

src_end = 16
src_num = 3

dport_end = 23
dport_num = 7

def get_encoding_mat(num_cats):
    idxs = np.arange(num_cats)
    lst = []
    
    num_freqs = num_cats // 2
    for i in range(num_freqs):
        angles = (2 * np.pi * idxs * (i+1)) / num_cats
        lst.append(np.sin(angles))
        lst.append(np.cos(angles))
    if num_cats % 2 != 0:
        lst.append(np.ones(num_cats))
    
    mat = np.stack(lst, axis=1)
    mat_inv = np.linalg.inv(mat)
    
    return mat, mat_inv

def circular_encode(data, mat):
    encoded = np.dot(data, mat)
    return encoded

def circular_decode(encoded, mat_inv):
    decoded = np.dot(encoded, mat_inv)
    return np.clip(np.round(decoded), 0, None)

def augment_cat_feature(normal, attack, num_cat):
    mat, mat_inv = get_encoding_mat(num_cat)
    encoded_normal = circular_encode(normal, mat)
    encoded_attack = circular_encode(attack, mat)
    summed = np.add(encoded_normal, encoded_attack)
    augmented = circular_decode(summed, mat_inv)
    return augmented

def augment_window(win_X, win_y, win_y_full):
    normal_idxs = np.argwhere(pd.isna(win_y_full))
    attack_idxs = np.argwhere(~pd.isna(win_y_full))
    
    augmented_win_X_lst = []
    augmented_win_y_lst = []
    augmented_win_y_full_lst = []

    for normal_idx in normal_idxs:
        for attack_idx in attack_idxs:
            normal_sec_X = win_X[normal_idx[0]]
            attack_sec_X = win_X[attack_idx[0]]
            
            normal_sec_X_numeric = normal_sec_X[:n_numeric]
            attack_sec_X_numeric = attack_sec_X[:n_numeric]
            augmented_X_numeric = np.add(normal_sec_X_numeric, attack_sec_X_numeric)
            
            normal_sec_X_cat = normal_sec_X[n_numeric:]
            attack_sec_X_cat = attack_sec_X[n_numeric:]
            
            normal_sec_X_proto = normal_sec_X_cat[:proto_end]
            attack_sec_X_proto = attack_sec_X_cat[:proto_end]
            augmented_X_proto = augment_cat_feature(normal_sec_X_proto, attack_sec_X_proto, proto_num)
            
            normal_sec_X_stos = normal_sec_X_cat[proto_end:stos_end]
            attack_sec_X_stos = attack_sec_X_cat[proto_end:stos_end]
            augmented_X_stos = augment_cat_feature(normal_sec_X_stos, attack_sec_X_stos, stos_num)
            
            normal_sec_X_src = normal_sec_X_cat[stos_end:src_end]
            attack_sec_X_src = attack_sec_X_cat[stos_end:src_end]
            augmented_X_src = augment_cat_feature(normal_sec_X_src, attack_sec_X_src, src_num)
            
            normal_sec_X_dport = normal_sec_X_cat[src_end:dport_end]
            attack_sec_X_dport = attack_sec_X_cat[src_end:dport_end]
            augmented_X_dport = augment_cat_feature(normal_sec_X_dport, attack_sec_X_dport, dport_num)
            
            augmented_sec_X = np.concat((augmented_X_numeric, augmented_X_proto, augmented_X_stos, augmented_X_src, augmented_X_dport)) 
            augmented_win_X = win_X.copy()
            augmented_win_X[normal_idx] = augmented_sec_X
            augmented_win_y = win_y
            augmented_win_y_full = win_y_full.copy()
            augmented_win_y_full[normal_idx] = augmented_win_y_full[attack_idx]

            augmented_win_X_lst.append(augmented_win_X)
            augmented_win_y_lst.append(augmented_win_y)
            augmented_win_y_full_lst.append(augmented_win_y_full)
    
    return augmented_win_X_lst, augmented_win_y_lst, augmented_win_y_full_lst

In [ ]:
augmented_X = []
augmented_y = []
augmented_y_full = []

for idx in idxs_to_augment:
    win_X = X_train[idx]
    win_y = y_train[idx]
    win_y_full = y_full_train[idx]
    augmented_win_X_lst, augmented_win_y_lst, augmented_win_y_full_lst = augment_window(win_X, win_y, win_y_full)
    augmented_X = augmented_X + augmented_win_X_lst
    augmented_y = augmented_y + augmented_win_y_lst
    augmented_y_full = augmented_y_full + augmented_win_y_full_lst

augmented_X = np.array(augmented_X)
augmented_y = np.array(augmented_y)
augmented_y_full = np.array(augmented_y_full)

In [ ]:
print(augmented_X.shape)

In [ ]:
augmented_X_train = np.concat((X_train, augmented_X))
augmented_y_train = np.concat((y_train, augmented_y))
augmented_pairs_train = np.concat((pairs_train, pairs_train[idxs_to_augment]))

In [ ]:
# NOTE: since feature_cols = numeric_features + categorical_features, we can rely on the indexing to be the same
n_numeric = len(numeric_features)

def scale_numeric(X, scaler, fit=False):
    X_numeric = X[:, :, :n_numeric].reshape(-1, n_numeric)
    X_numeric = scaler.fit_transform(X_numeric) if fit else scaler.transform(X_numeric)
    X[:, :, :n_numeric] = X_numeric.reshape(X.shape[0], X.shape[1], n_numeric)
    return X
    
scaler = StandardScaler()
#X_train = scale_numeric(X_train, scaler, fit=True)
augmented_X_train = scale_numeric(augmented_X_train, scaler, fit=True)
X_val = scale_numeric(X_val, scaler)
X_test = scale_numeric(X_test, scaler)

In [ ]:
np.unique(y_train)

In [ ]:
# index 0 represents backdoor, 1 represents comminj, and so on
CLASSES = ['Backdoor', 'CommInj', 'DoS', 'Reconn', 'normal']

#y_train = label_binarize(y_train, classes=CLASSES)
augmented_y_train = label_binarize(augmented_y_train, classes=CLASSES)
y_val = label_binarize(y_val, classes=CLASSES)
y_test = label_binarize(y_test, classes=CLASSES)

In [ ]:
np.save('../../data/processed/augmented_X_train.npy', augmented_X_train)
np.save('../../data/processed/X_val.npy', X_val)
np.save('../../data/processed/X_test.npy', X_test)

np.save('../../data/processed/augmented_pairs_train.npy', augmented_pairs_train)
np.save('../../data/processed/pairs_val.npy', pairs_val)
np.save('../../data/processed/pairs_test.npy', pairs_test)

np.save('../../data/processed/augmented_y_train.npy', augmented_y_train)
np.save('../../data/processed/y_val.npy', y_val)
np.save('../../data/processed/y_test.npy', y_test)